In [ ]:
# General library imports.
import sys
from pathlib import Path

# Find the example helpers from either the repository root or this directory.
notebook_dir = Path.cwd() if (Path.cwd() / "text_abstention.py").exists() else Path.cwd() / "quickstart_ipynbs"
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

%load_ext autoreload
%autoreload 2
%aimport -torch -sklearn -numpy -np -pandas -tensorflow -transformers -trl

import json
from itertools import islice
import torch
import pandas as pd
from IPython.display import display
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from outlines import generate, models, samplers
from text_abstention import FLAG_QUESTIONS, text_abstained, summarize_predictions

In [ ]:
# Setup in a fresh Python 3.11/3.12 environment, then restart the kernel:
# %pip install torch torchvision "transformers==4.51.3" "outlines==0.1.14" "datasets==4.0.0" "huggingface-hub==0.36.0" "accelerate==1.6.0" pillow pandas sentencepiece
# nlg2choice is released as example code, not as a pip package named nlg2choice.
# This uses its public generate.choice / models.Transformers implementation:
# https://github.com/Racerx250/nlg2choice/blob/main/tasks/260720_1_model_examples/example_saved_nlg2choice.ipynb
# Reference setup: https://github.com/Racerx250/nlg2choice/blob/main/environment.yml
# Model: https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct
# Set HF_HOME to a data/cache drive before starting Jupyter on shared systems.

REPO_ID = "cvl-umass/RealBirdID"
DATA_REVISION = "abd12f9a663f47bc64b8345eeb54679d134762aa"  # Pin dataset and model commits for reproducible results.
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
MODEL_REVISION = "main"
N_PER_SUBSET = 2
QUESTION = "What is the species of this bird?"
MAX_NEW_TOKENS = 128
assert torch.cuda.is_available(), "Run this notebook in a CUDA GPU runtime."
# One model, one image at a time, at most 256 visual tokens; no CPU offload.
# CUDA builds must support the assigned GPU; this check runs before model loading.
print("GPU:", torch.cuda.get_device_name(0), "capability:", torch.cuda.get_device_capability(0))
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda, "build architectures:", torch.cuda.get_arch_list())
assert torch.ones(1, device="cuda").sum().item() == 1


In [ ]:
# Load a tiny sample. This is a workflow demonstration, not a paper reproduction.
examples = []
for subset in ("answerable", "unanswerable"):
    stream = load_dataset(REPO_ID, subset, split="validation", revision=DATA_REVISION, streaming=True)
    examples.extend({"subset": subset, **row} for row in islice(stream, N_PER_SUBSET))
assert len(examples) == 2 * N_PER_SUBSET
for row in examples:
    print(row["subset"], row["observation_id"], row["license"], row["photo_attribution"])
    print(row["source_url"])
# All source license values are included. Report filters if you change this scope.

In [ ]:
# Use the released species vocabulary, never the labels of the evaluation sample.
taxonomy_path = hf_hub_download(REPO_ID, "taxonomy.json", repo_type="dataset", revision=DATA_REVISION)
taxonomy = json.loads(Path(taxonomy_path).read_text())
choice_list = sorted({taxon["name"] for taxon in taxonomy if taxon["rank"] == "species"})
assert choice_list
assert all(row["taxon_name"] in choice_list for row in examples if row["subset"] == "answerable")
print("Species choices:", len(choice_list))

In [ ]:
# Load the small Qwen model once for vision generation and text-only extraction.
torch.manual_seed(42)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
processor = AutoProcessor.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, min_pixels=256 * 28 * 28, max_pixels=256 * 28 * 28,
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, torch_dtype=dtype,
    device_map={"": 0}, attn_implementation="sdpa",
).eval()
model.generation_config.pad_token_id = processor.tokenizer.pad_token_id

In [ ]:
def generate_response(image):
    messages = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": QUESTION},
    ]}]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt], images=[image.convert("RGB")], return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    # Decode only new tokens; the question must not enter the extracted response.
    new_tokens = output[:, inputs["input_ids"].shape[1]:]
    text = processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()
    assert text, "The model returned an empty response."
    return text

for row in examples:
    row["response"] = generate_response(row["image"])
display(pd.DataFrame([{key: row[key] for key in ("subset", "observation_id", "response")} for row in examples]))

In [ ]:
# nlg2choice: constrain text decoding to exactly one released species label.
# Labels constrain the decoder; the full list is not copied into the prompt.
# This keeps the prompt small and does not use a per-example ground truth label.
text_model = models.Transformers(model, processor.tokenizer)
species_decoder = generate.choice(text_model, choice_list, sampler=samplers.greedy())
yesno_decoder = generate.choice(text_model, ["Yes", "No"], sampler=samplers.greedy())

def text_prompt(question):
    return processor.apply_chat_template(
        [{"role": "user", "content": [{"type": "text", "text": question}]}],
        tokenize=False, add_generation_prompt=True,
    )

def extract_species(response):
    prompt = f"What bird species does this response identify?\nResponse:\n{response}\nReturn the species label only."
    with torch.inference_mode():
        prediction = species_decoder(text_prompt(prompt), max_tokens=64)
    if prediction not in choice_list:
        raise ValueError(f"Incomplete species extraction: {prediction!r}")
    return prediction

def extract_flags(response):
    extracted = {}
    for key, question in FLAG_QUESTIONS.items():
        target = "No" if key == "is_prediction_ambiguous" else "Yes"
        prompt = (
            "Judge the supplied response, not the bird image. Do not answer the bird question yourself.\n"
            f"Response:\n{response}\nQuestion about the response: {question}\n"
            f"Does the response support the answer '{target}' to this question? Answer Yes or No."
        )
        with torch.inference_mode():
            extracted[key] = yesno_decoder(text_prompt(prompt), max_tokens=8)
    # Validate every flag before calculating abstention.
    text_abstained(extracted)
    return extracted
# The small Qwen judge illustrates constrained extraction. The original evaluation
# used xVerify verification; judge errors can change the flags and the measured rate.

In [ ]:
# Keep response, flags, extracted species, and ground truth together for inspection.
results = []
for row in examples:
    extracted_flags = extract_flags(row["response"])
    abstained = text_abstained(extracted_flags)
    prediction = None if abstained else extract_species(row["response"])
    results.append({
        "observation_id": row["observation_id"], "subset": row["subset"],
        "ground_truth": row["taxon_name"], "ground_truth_rank": row["taxon_rank"],
        "response": row["response"], "prediction": prediction, "abstained": abstained,
        "license": row["license"], **extracted_flags,
    })
display(pd.DataFrame(results))
display(pd.Series(summarize_predictions(results), name="small-sample metrics"))
# No species accuracy is computed on UA: its taxon label is not a species answer.
# Small-sample text rates do not reproduce the paper's retrieval probabilities,
# taxonomy aggregation, threshold-swept UA/A, IG, or abstention-reason accuracy.